# BM25 then select multiple categories with Jev

This notebook recreates [`ecom_query_understanding_category_hierarchy_bm25_boosted_jev_bm25_then_select_multiple.yml`](../configs/ecom_class/full/jev/ecom_query_understanding_category_hierarchy_bm25_boosted_jev_bm25_then_select_multiple.yml) on WANDS. First, a dedicated BM25 search finds candidate category paths from the top 1,000 positive-score matches. Jev selects every candidate with probability strictly above `0.4`. The query-understanding strategy then runs a second BM25 search and boosts documents matching any selected category.

The full CLI command is:

```bash
uv run run --strategy configs/ecom_class/full/jev/ecom_query_understanding_category_hierarchy_bm25_boosted_jev_bm25_then_select_multiple.yml --dataset wands --workers 4 --no-cache
```

## Install pinned dependencies

`cheat-at-search` provides WANDS data, SearchStrategy, and evaluation helpers. The TypeSafe SDK provides Jev Choice.

In [1]:
!pip install -q git+https://github.com/softwaredoug/cheat-at-search.git@1ee82acd943135997b625846525e7f44329f55c8
!pip install -q typesafe-sdk==0.7.1


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Mount the shared data directory

Mount Google Drive in Colab to retain WANDS data and API credentials between sessions. Outside Colab, use the local cache directory.

In [2]:
from pathlib import Path

from cheat_at_search.data_dir import mount

try:
    mount(use_gdrive=True)
except ImportError:
    manual_path = str(Path.home() / '.search-experiments' / 'cheat-at-search')
    mount(use_gdrive=False, manual_path=manual_path)

2026-10-06 16:16:52,052 - cheat_at_search.data_dir - ERROR - Google Colab drive module not found. Ensure you're running this in Google Colab.


## Load the Jev key and WANDS data

WANDS contains product titles, descriptions, full category paths, and query relevance judgments.

In [3]:
import hashlib
import json
import math

import numpy as np
import pandas as pd

from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus as wands_corpus
from cheat_at_search.wands_data import judgments as wands_judgments
from typesafe_sdk import Choice, TypeSafeClient

TYPESAFE_KEY = key_for_provider('typesafe')
corpus = wands_corpus.reset_index(drop=True).copy()
judgments = wands_judgments.reset_index(drop=True).copy()
CATEGORY_FIELD = 'category hierarchy'

corpus[['doc_id', 'title', CATEGORY_FIELD]].head(3)

,doc_id,title,category hierarchy
0,0,solid wood platform bed,Furniture / Bedroom Furniture / Beds & Headboa...
1,1,all-clad 7 qt . slow cooker,Kitchen & Tabletop / Small Kitchen Appliances ...
2,2,all-clad electrics 6.5 qt . slow cooker,Kitchen & Tabletop / Small Kitchen Appliances ...


## Build the candidate BM25 search

This search is independent of the final retrieval engine. It scores the configured text fields, drops score-zero documents, takes the top 1,000 matches, and counts the category paths in those rows. The 255 most frequent paths become Jev's options; frequency ties follow the DataFrame's `value_counts` ordering.

In [4]:
from searcharray import SearchArray
from searcharray.similarity import bm25_similarity

from cheat_at_search.strategy import SearchStrategy
from cheat_at_search.tokenizers import snowball_tokenizer

CANDIDATE_FIELDS = {'title': 9.4, 'description': 4.0}
FINAL_FIELDS = {'title': 9.4, 'description': 4.0}
AGGREGATE_OVER = 1000
JEV_OPTION_LIMIT = 255
THRESHOLD = 0.4
BOOST_MATCHES = 10
BM25_K1 = 1.2
BM25_B = 0.75
SIMILARITY = bm25_similarity(k1=BM25_K1, b=BM25_B)
PROMPT_TEMPLATE = (
    'Below is a search query for a furniture / homegoods store. Select all\n'
    'relevant values from the category hierarchy. Category values use /\n'
    'separators; return the full path for each relevant category.\n\n'
    'Query:\n{query}\n'
)

for field in {*CANDIDATE_FIELDS, *FINAL_FIELDS, CATEGORY_FIELD}:
    index_name = f'{field}_snowball'
    if index_name not in corpus.columns:
        corpus[index_name] = SearchArray.index(
            corpus[field].fillna('').astype(str), snowball_tokenizer
        )

def bm25_scores(query, field_weights):
    scores = np.zeros(len(corpus), dtype=float)
    for term in snowball_tokenizer(query):
        for field, weight in field_weights.items():
            scores += (
                corpus[f'{field}_snowball'].array.score(
                    term, similarity=SIMILARITY
                )
                * weight
            )
    return scores

def candidate_category_counts(query):
    scores = bm25_scores(query, CANDIDATE_FIELDS)
    positive_indices = np.flatnonzero(scores > 0)
    order = np.argsort(-scores[positive_indices])[:AGGREGATE_OVER]
    matched_rows = corpus.iloc[positive_indices[order]]
    values = matched_rows[CATEGORY_FIELD].dropna().astype(str)
    values = values[values.str.strip() != '']
    return values.value_counts()

example_query = judgments['query'].drop_duplicates().iloc[0]
candidate_counts = candidate_category_counts(example_query)
candidate_counts.head(JEV_OPTION_LIMIT).head(10)

2026-10-06 16:16:56,454 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-10-06 16:16:56,456 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-10-06 16:16:56,456 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

2026-10-06 16:16:56,859 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-10-06 16:16:56,866 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


category hierarchy
Furniture / Office Furniture / Office Chairs                                                                                 311
Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen & Dining Chairs                                   182
Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen & Dining Chairs / Side Kitchen & Dining Chairs     63
Baby & Kids / Toddler & Kids Playroom / Playroom Furniture / Toddler & Kids Chairs & Seating                                  48
Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio Seating / Outdoor Club Chairs                    46
Furniture / Living Room Furniture / Chairs & Seating / Accent Chairs                                                          43
Outdoor / Outdoor & Patio Furniture / Outdoor Seating & Patio Chairs / Patio Seating / Patio Rocking Chairs & Gliders         33
Furniture / Living Room Furniture / Chairs & Seating / Massage Chairs         

## Ask Jev to select multiple candidate paths

The Choice criteria contain only paths found by candidate retrieval. No synthetic `Unknown` is added. A category is kept only if its probability is strictly greater than `0.4`; a query with no candidate matches returns an empty category list without calling Jev.

In [5]:
jev = TypeSafeClient(api_key=TYPESAFE_KEY, model='jev-latest')
classification_cache = {}

def classify_query(query, client=jev):
    candidates = candidate_category_counts(query).head(JEV_OPTION_LIMIT).index.tolist()
    if not candidates:
        return {'candidate_categories': [], 'categories': [], 'probabilities': {}}

    cache_key = (query, tuple(candidates))
    if cache_key not in classification_cache:
        response = client.system_one(
            state=query,
            questions={
                CATEGORY_FIELD: Choice(
                    instructions=PROMPT_TEMPLATE.format(query=query),
                    criteria={category: None for category in candidates},
                )
            },
        )
        probabilities = dict(
            getattr(response.choices[CATEGORY_FIELD], 'probabilities', {}) or {}
        )
        selected = sorted(
            {
                category
                for category, probability in probabilities.items()
                if category in candidates
                and category != 'Unknown'
                and isinstance(probability, (int, float))
                and math.isfinite(probability)
                and probability > THRESHOLD
            }
        )
        classification_cache[cache_key] = {
            'candidate_categories': candidates,
            'categories': selected,
            'probabilities': probabilities,
        }
    return classification_cache[cache_key]

example_classification = classify_query(example_query)
example_classification['categories'], (
    pd.Series(example_classification['probabilities'], name='probability')
    .sort_values(ascending=False)
    .head(10)
)

(['Shop Product Type / Chairs / Guest & Reception Chairs'],
 Shop Product Type / Chairs / Guest & Reception Chairs                                                                                       0.55
 Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs                                          0.18
 Furniture / Living Room Furniture / Chairs & Seating / Massage Chairs                                                                       0.13
 Furniture / Living Room Furniture / Chairs & Seating / Accent Chairs                                                                        0.06
 Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs / Wood Waiting Room Chairs               0.03
 Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs / Metal Waiting Room Chairs              0.02
 Furniture / Living Room Furniture / Chairs & Seating / Accent C

## Run the second BM25 search with category boosts

After classification, the final search computes ordinary BM25 scores again. Each selected full category path adds `10` to documents whose category hierarchy matches that path. If multiple selected paths match one document, their boosts add.

In [6]:
class JevBM25ThenSelectMultiple(SearchStrategy):
    def __init__(self, corpus, workers=4):
        super().__init__(corpus, top_k=10, workers=workers)
        self.index = corpus
        self.category_index_name = f'{CATEGORY_FIELD}_snowball'

    def _category_matches(self, category):
        terms = snowball_tokenizer(category)
        if not terms:
            return np.zeros(len(self.index), dtype=bool)
        return self.index[self.category_index_name].array.score(terms) > 0

    def search(self, query, k=10):
        # First BM25 search -> Jev classification.
        categories = classify_query(query)['categories']

        # Second BM25 search -> apply the selected category boosts.
        scores = bm25_scores(query, FINAL_FIELDS)
        for category in categories:
            scores[self._category_matches(category)] += BOOST_MATCHES

        top_indices = np.argsort(-scores)[:k]
        return top_indices, scores[top_indices]

    @property
    def cache_key(self):
        payload = {
            'type': 'jev_bm25_then_select_multiple',
            'category_field': CATEGORY_FIELD,
            'candidate_fields': CANDIDATE_FIELDS,
            'final_fields': FINAL_FIELDS,
            'aggregate_over': AGGREGATE_OVER,
            'threshold': THRESHOLD,
            'prompt': PROMPT_TEMPLATE,
            'model': 'jev-latest',
            'boost_matches': BOOST_MATCHES,
            'k1': BM25_K1,
            'b': BM25_B,
        }
        encoded = json.dumps(payload, sort_keys=True).encode('utf-8')
        return hashlib.md5(encoded).hexdigest()

strategy = JevBM25ThenSelectMultiple(corpus, workers=4)
strategy.search(example_query, k=5)

(array([ 7465, 25431,  7468, 39461,  9234]),
 array([75.86080217, 67.17602205, 63.4888978 , 58.95511341, 57.55410337]))

## Inspect ranked results

This query reuses its Jev classification from the in-memory cache. Scores include both text relevance and the boosts for selected category paths.

In [7]:
top_indices, top_scores = strategy.search(example_query, k=5)
preview = corpus.iloc[top_indices][['doc_id', 'title', CATEGORY_FIELD]].copy()
preview['score'] = top_scores
preview

,doc_id,title,category hierarchy,score
7465,7465,hair salon chair,Furniture / Living Room Furniture / Chairs & S...,75.860802
25431,25431,barberpub salon massage chair,Furniture / Living Room Furniture / Chairs & S...,67.176022
7468,7468,mercer41 hair salon chair hydraulic styling ch...,Furniture / Living Room Furniture / Chairs & S...,63.488898
39461,39461,professional salon reclining massage chair,Furniture / Living Room Furniture / Chairs & S...,58.955113
9234,9234,beauty salon task chair,Furniture / Office Furniture / Office Chairs,57.554103


## Compare against baseline BM25

`run_strategy` evaluates the ranked-search interface against WANDS relevance judgments. This notebook defaults to a small sample to limit Jev calls; set `QUERY_LIMIT = None` to run every query. The CLI command above uses the full dataset with four workers and disables the search-result cache.

In [8]:
from cheat_at_search.search import mrrs, ndcgs, run_strategy
from cheat_at_search.strategy import BM25Search

baseline = BM25Search(corpus, title_boost=9.4, description_boost=4.0)
baseline.workers = 4
baseline_graded = run_strategy(
    baseline, judgments, cache=False
)
boosted_graded = run_strategy(
    strategy, judgments, cache=False
)

metrics = pd.DataFrame(
    {
        'BM25 baseline': [ndcgs(baseline_graded).mean(), mrrs(baseline_graded).mean()],
        'Jev BM25 then select': [ndcgs(boosted_graded).mean(), mrrs(boosted_graded).mean()],
    },
    index=['Mean NDCG@10', 'Mean MRR'],
)
metrics

Searching: 100%|████████████████████████████████████████████████████████| 480/480 [00:23<00:00, 20.09it/s]


,BM25 baseline,Jev BM25 then select
Mean NDCG@10,0.540573,0.566768
Mean MRR,0.526878,0.548719


## What this experiment tests

The BM25 candidate search narrows Jev's options to categories represented in positive-score matches, prioritizing the most frequent paths. Jev can then select several categories, and the final BM25 search boosts documents matching any selected path. The comparison with baseline BM25 uses NDCG@10 and MRR.